# Imports

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Useful functions

In [ ]:
def permutation_test(scores_ctr, scores_h3, n_shuffle=10000, rng=None):
    """Permutation test for difference in means between two groups.

    Randomly reassigns group labels n_shuffle times and records the
    difference in group means (h3 - ctr) each time to build a null
    distribution.  Returns the null distribution and a two-tailed p-value.
    """
    if rng is None:
        rng = np.random.default_rng()

    scores_ctr = np.array(scores_ctr)
    scores_h3  = np.array(scores_h3)
    n_ctr = len(scores_ctr)
    all_scores = np.concatenate([scores_ctr, scores_h3])
    n_total = len(all_scores)

    observed_diff = np.nanmean(scores_h3) - np.nanmean(scores_ctr)

    null_diffs = np.empty(n_shuffle)
    for i in range(n_shuffle):
        perm = rng.permutation(n_total)
        null_diffs[i] = np.nanmean(all_scores[perm[n_ctr:]]) - np.nanmean(all_scores[perm[:n_ctr]])

    # Two-tailed p-value
    p_val = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
    return observed_diff, null_diffs, p_val

In [ ]:
def plot_silhouette_stats(scores_ctr, scores_h3, observed_diff, null_diffs, p_val, fig_path):
    """Two-panel figure: group scatter + permutation null distribution."""

    scores_ctr = np.array(scores_ctr)
    scores_h3  = np.array(scores_h3)

    colors = {'ctr': 'orangered', 'h3': 'deepskyblue'}

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    # --- Panel 1: individual scores + mean +/- SEM per group ---
    ax = axs[0]
    for x_pos, (label, scores, color) in enumerate(
        [('ctr', scores_ctr, colors['ctr']), ('h3', scores_h3, colors['h3'])]
    ):
        jitter = np.random.default_rng(0).uniform(-0.1, 0.1, size=len(scores))
        ax.scatter(np.full(len(scores), x_pos) + jitter, scores,
                   color=color, s=60, zorder=3, alpha=0.8)
        mean = np.nanmean(scores)
        sem  = np.nanstd(scores) / np.sqrt(np.sum(~np.isnan(scores)))
        ax.errorbar(x_pos, mean, yerr=sem, fmt='o', color='black',
                    markersize=8, linewidth=2, zorder=4)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Control', 'H3'])
    ax.set_ylabel('Silhouette score')
    ax.set_title('Silhouette scores per group')
    ax.spines[['top', 'right']].set_visible(False)

    # --- Panel 2: permutation null distribution ---
    ax = axs[1]
    ax.hist(null_diffs, bins=50, color='gray', edgecolor='white', alpha=0.7,
            label='Null distribution')
    ymax = ax.get_ylim()[1]

    # 95% CI thresholds (two-tailed, i.e. 2.5th and 97.5th percentiles)
    lo, hi = np.percentile(null_diffs, [2.5, 97.5])
    ax.vlines([lo, hi], 0, ymax, color='red', linewidth=2,
              linestyle='--', label='95% CI (null)')
    ax.vlines(observed_diff, 0, ymax, color='green', linewidth=3,
              label=f'Observed diff = {observed_diff:.4f}')

    sig_str = 'SIGNIFICANT' if p_val < 0.05 else 'not significant'
    ax.set_title(f'Permutation test  -  p = {p_val:.4f}  ({sig_str})')
    ax.set_xlabel('H3 mean - Control mean (silhouette)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

    fig.tight_layout()
    fig.savefig(fig_path, format='png', bbox_inches='tight', dpi=100)
    plt.show()
    print(f'Saved -> {fig_path}')

# Datafolder definition

In [ ]:
# ============ CHANGE THESE ============
main_dir   = '/home/user/Rotarod'
pickle_dir = main_dir + '/results/pickle/'
plots_dir  = main_dir + '/results/plots/'
# ======================================

# Main code

In [ ]:
# Load results saved by Compute_silhouette_plot_selected.ipynb
file_to_open = pickle_dir + 'ctr_h3_silh_results.pickle'
results_sil  = pickle.load(open(file_to_open, 'rb'))

ctr_h3_silh_dict = results_sil['ctr_h3_silh_dict']

scores_ctr = ctr_h3_silh_dict['ctr']   # list of one score per control mouse
scores_h3  = ctr_h3_silh_dict['h3']    # list of one score per h3 mouse

print(f'Control  ({len(scores_ctr)} mice): {[round(s, 4) for s in scores_ctr]}')
print(f'H3       ({len(scores_h3)}  mice): {[round(s, 4) for s in scores_h3]}')

In [ ]:
# Run permutation test
n_shuffle = 10000
rng = np.random.default_rng()

observed_diff, null_diffs, p_val = permutation_test(
    scores_ctr, scores_h3, n_shuffle=n_shuffle, rng=rng
)

print(f'Observed difference (H3 - Control): {observed_diff:.4f}')
print(f'p-value (two-tailed, n={n_shuffle} permutations): {p_val:.4f}')

In [ ]:
# Plot and save
fig_path = plots_dir + 'silhouette_permtest.png'
plot_silhouette_stats(
    scores_ctr, scores_h3, observed_diff, null_diffs, p_val, fig_path
)